In [57]:
%pip install -q bertopic openai sentence-transformers pandas

In [91]:
import pandas as pd

reviews_df = pd.read_csv('raw_intents_output.csv')
reviews_df = reviews_df.dropna(subset=["intent", "text_summary"])
reviews_df.head()


,id,Review,llm_response,intent,text_summary,sentiment,named_entities
0,0,Tel Aviv to Amman. We had a short flight to ...,"{'intent': 'Flight route and connections', 'na...",Flight route and connections,"Tel Aviv to Amman, connecting to Heathrow and ...",Neutral,[]
1,0,Tel Aviv to Amman. We had a short flight to ...,"{'intent': 'In-flight smoking issue', 'named_e...",In-flight smoking issue,There was smoking on the plane,Negative,[]
2,0,Tel Aviv to Amman. We had a short flight to ...,"{'intent': 'Flight attendant service', 'named_...",Flight attendant service,The stewardess spent the entire time on her ce...,Negative,[]
3,1,Flight got delayed when I was flying to DR f...,"{'intent': 'Flight delay', 'named_entities': [...",Flight delay,Flight got delayed for an hour,Negative,['Frontier']
4,1,Flight got delayed when I was flying to DR f...,"{'intent': 'Flight schedule change', 'named_en...",Flight schedule change,Flight got advanced by 30 minutes,Neutral,['Frontier']


In [92]:
all_reviews = [f"{row['intent']}: {row['text_summary']}" for _, row in reviews_df.iterrows()]
len(all_reviews)


101

In [93]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
embeddings = embedding_model.encode(all_reviews, show_progress_bar=True)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [96]:
embedding_file_path = "embeddings_reviews.npy"
import numpy as np
with open(embedding_file_path, "wb") as f:
    np.save(f, embeddings)


In [97]:
predefined_topics = [
    "Check-in and Boarding",
    "Seating Comfort",
    "In-Flight Wi-Fi",
    "Cabin Cleanliness",
    "Food and Beverage",
    "Flight Attendants and Crew Services",
    "Baggage Handling",
    "Flight Disruptions and Delays",
    "Loyalty Program and benefits",
    "Pricing Transparency and Fees",
    "Safety Measures",
    "Ground Services Assistance",
    "Accessibility and special assistance",
]
len(predefined_topics)


13

In [98]:
import openai

client = openai.OpenAI(api_key=OPENAI_API_KEY)


### Create the representation model

In [99]:
prompt = """
Generate a short topic label based on reviews and key words describing the reviews.

Make sure the response only contains the topic label. The response should be in the following format:
'''
topic:
'''

Here are some examples of topic labels:
'''
topic: Seating Comfort
topic: Food and Beverage Quality
topic: Baggage Handling
'''

Here are the reviews:
[DOCUMENTS]

Here are the keywords that are relevant to the reviews. Use them as reference for the topic label, but keep in mind the key words are not always the most representative. [KEYWORDS]

Now read the reviews and keywords carefully, and respond with the topic label that best categories the reviews. Remember to respond in the correct format.
"""


In [100]:
from bertopic.representation import OpenAI

openai_generator = OpenAI(
    client,
    model=MODEL_NAME,
    chat=True,
    nr_docs=5,
    prompt=prompt,
)


In [101]:
print(openai_generator.default_prompt_)

You will extract a short topic label from given documents and keywords.
Here are two examples of topics you created before:

# Example 1
Sample texts from this topic:
- Traditional diets in most cultures were primarily plant-based with a little meat on top, but with the rise of industrial style meat production and factory farming, meat has become a staple food.
- Meat, but especially beef, is the worst food in terms of emissions.
- Eating meat doesn't make you a bad person, not eating meat doesn't make you a good one.

Keywords: meat beef eat eating emissions steak food health processed chicken
topic: Environmental impacts of eating meat

# Example 2
Sample texts from this topic:
- I have ordered the product weeks ago but it still has not arrived!
- The website mentions that it only takes a couple of days to deliver but I still have not received mine.
- I got a message stating that I received the monitor but that is not true!
- It took a month longer to deliver than was advised...

Key

In [102]:
print(openai_generator.prompt)


Generate a short topic label based on reviews and key words describing the reviews.

Make sure the response only contains the topic label. The response should be in the following format:
'''
topic:
'''

Here are some examples of topic labels:
'''
topic: Seating Comfort
topic: Food and Beverage Quality
topic: Baggage Handling
'''

Here are the reviews:
[DOCUMENTS]

Here are the keywords that are relevant to the reviews. Use them as reference for the topic label, but keep in mind the key words are not always the most representative. [KEYWORDS]

Now read the reviews and keywords carefully, and respond with the topic label that best categories the reviews. Remember to respond in the correct format.



In [103]:
from bertopic import BERTopic


def fixed_topic_labels_(self):
    """Map topic IDs to their labels.
    A label is the topic ID, along with the first four words of the topic representation, joined using '_'.
    Zeroshot topic labels come from self.zeroshot_topic_list rather than the calculated representation.
    """
    topic_labels = {
        key: f"{key}_" + "_".join([word[0] for word in values[:4]])
        for key, values in self.topic_representations_.items()
    }
    if self._is_zeroshot():
        topic_id_to_zeroshot_label = {
            self.topic_mapper_.get_mappings()[topic_id]: self.zeroshot_topic_list[zeroshot_topic_idx]
            for topic_id, zeroshot_topic_idx in self._topic_id_to_zeroshot_topic_idx.items()
        }
        topic_labels.update(topic_id_to_zeroshot_label)
    return topic_labels


BERTopic.topic_labels_ = property(fixed_topic_labels_)


In [104]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

ai_representation = [MaximalMarginalRelevance(diversity=0.3), openai_generator]

representations = {
    "AI_Generated": ai_representation,
    "KeyBERT": KeyBERTInspired(),
}


In [105]:
topic_model = BERTopic(
    embedding_model="BAAI/bge-base-en-v1.5",
    verbose=True,
    min_topic_size=5,
    zeroshot_topic_list=predefined_topics,
    zeroshot_min_similarity=0.6,
    representation_model=representations,
)

In [106]:
min_topic_size=5

In [107]:
print(len(all_reviews))

101


In [108]:
SAMPLE_SIZE = 100

In [154]:
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(min_cluster_size=2, min_samples=1, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

topic_model = BERTopic(
    embedding_model="BAAI/bge-base-en-v1.5",
    verbose=True,
    hdbscan_model=hdbscan_model,
    zeroshot_topic_list=predefined_topics,
    zeroshot_min_similarity=0.6,
    representation_model=representations,
)

In [131]:
from bertopic import BERTopic


topic_model = BERTopic()


topics, probs = topic_model.fit_transform(all_reviews, embeddings)
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,28,-1_the_and_was_seats,"[the, and, was, seats, were, no, flight, seat,...",[Preferred seat purchase: We purchased preferr...
1,0,62,0_the_flight_to_and,"[the, flight, to, and, was, experience, on, wi...",[Flight cancellation: The flight was canceled ...
2,1,11,1_customer_service_refund_and,"[customer, service, refund, and, the, they, to...",[Customer service experience: We called custom...


In [130]:
# Updated topics
topic_model.get_topic_info()


,Topic,Count,Name,Representation,AI_Generated,KeyBERT,Representative_Docs
0,-1,8,-1_smoking_complete_delta_plane,"[smoking, complete, delta, plane, policy, for,...",[Airline Customer Service],"[cancelations, flight, cancellation, fare, lax...",[Delta's assistance: Delta helped get my child...
1,0,8,Check-in and Boarding,"[check, boarding, in, board, and, coffee, expe...",[Check-in and Boarding Experience],"[boarding, booked, flight, checking, booking, ...",[Buy on board service: There was however a buy...
2,1,5,Seating Comfort,"[comfort, seats, seat, spacing, seating, comfo...",[Seating Discomfort],"[seats, seating, seat, cramped, comfort, comfo...","[Seat comfort: seats are comfortable enough, S..."
3,2,1,In-Flight Wi-Fi,"[fine, mvd, sao, tam, board, with, flight, on,...",[In-Flight Experience],"[flight, tam, sao, mvd, fine, experience, boar...",[In-flight experience with TAM: Flight MVD to ...
4,3,3,Cabin Cleanliness,"[cabin, airplane, temperature, about, a319, cl...",[Cabin Environment],"[cabin, airplane, cleanliness, flight, passeng...",[Airplane maintenance: Well maintained and cle...
5,4,5,Flight Attendants and Crew Services,"[crew, service, cabin, challenges, condescendi...",[Crew Service Quality],"[crew, service, flight, cabin, response, chall...",[Cabin Crew service: The Cabin Crew were very ...
6,5,3,Baggage Handling,"[bags, baggage, wet, contents, frontier, every...",[Baggage Handling Issues],"[baggage, bags, misplaced, delivered, flown, l...",[Customer service: The rep said that the airli...
7,6,22,Flight Disruptions and Delays,"[flight, the, to, was, cancellation, delayed, ...",[Flight Reliability],"[delays, delayed, cancellations, cancellation,...",[Waiting time and contradictory information: T...
8,7,4,Pricing Transparency and Fees,"[fees, hidden, price, preferred, booking, tick...",[Booking and Payment Transparency],"[ticket, fees, booking, transparency, charged,...",[Transparency of policies: The customer feels ...
9,8,2,Safety Measures,"[safety, covid, 19, journey, distancing, throu...",[In-flight Covid-19 Safety Measures],"[safety, safe, distancing, journey, covid, thr...","[Safety: i felt safe throughout the journey, C..."


In [139]:
topic_names = [topic_model.get_topic_info(topic_id)["Name"].iloc[0] for topic_id in topics]

# Correctly retrieve LLM-generated topic names from topic_model.topic_aspects_
llm_topic_names = []
for topic_id in topics:
    if topic_id == -1:
        llm_topic_names.append("Outlier Topic")
    else:
        # Check if the topic_id exists in topic_model.topic_aspects_ and has an 'AI_Generated' aspect
        if topic_id in topic_model.topic_aspects_ and "AI_Generated" in topic_model.topic_aspects_[topic_id]:
            # Access the AI_Generated aspect, which is a list of (label, probability) tuples
            llm_label_info = topic_model.topic_aspects_[topic_id]["AI_Generated"][0][0]
            llm_topic_names.append(llm_label_info.strip("'''").strip("\n"))
        else:
            # Fallback: if AI_Generated representation is missing, use the default BERTopic name
            default_name_df = topic_model.get_topic_info(topic_id)
            default_name = default_name_df["Name"].iloc[0]
            # Remove the numerical prefix from BERTopic's default name if present
            if '_' in default_name and default_name.split('_')[0].isdigit():
                default_name = '_'.join(default_name.split('_')[1:])
            llm_topic_names.append(f"BERTopic Default: {default_name}")

results_df = pd.DataFrame(
    data={
        "topic_id": topics,
        "probability": probs,
        "topic_name": topic_names,
        "llm_topic_name": llm_topic_names,
        "document": all_reviews,
    }
)
results_df

,topic_id,probability,topic_name,llm_topic_name,document
0,19,0.892305,19_connecting_aviv_amman_connections,BERTopic Default: connecting_aviv_amman_connec...,Flight route and connections: Tel Aviv to Amma...
1,-1,0.696162,-1_af_supervisor_smoking_sales,Outlier Topic,In-flight smoking issue: There was smoking on ...
2,10,0.816986,10_amenities_announcement_bottles_spent,BERTopic Default: amenities_announcement_bottl...,Flight attendant service: The stewardess spent...
3,6,0.870428,Flight Disruptions and Delays,BERTopic Default: Flight Disruptions and Delays,Flight delay: Flight got delayed for an hour
4,6,0.830728,Flight Disruptions and Delays,BERTopic Default: Flight Disruptions and Delays,Flight schedule change: Flight got advanced by...
...,...,...,...,...,...
96,6,0.835385,Flight Disruptions and Delays,BERTopic Default: Flight Disruptions and Delays,Airport experience: The airport experience was...
97,0,0.866019,Check-in and Boarding,BERTopic Default: Check-in and Boarding,Check-in and boarding: The check-in and boardi...
98,10,0.865329,10_amenities_announcement_bottles_spent,BERTopic Default: amenities_announcement_bottl...,Flight experience: The flight itself was quiet...
99,4,0.886608,Flight Attendants and Crew Services,BERTopic Default: Flight Attendants and Crew S...,Crew service: The crew was great despite the c...


In [140]:
results_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Saved {len(results_df)} rows to {OUTPUT_CSV_PATH}")


Saved 101 rows to labeled_reviews_output.csv


## Visualize topics

In [141]:
llm_topic_labels = {
    topic: values[0][0].strip("'''").strip("\n")
    for topic, values in topic_model.topic_aspects_["AI_Generated"].items()
}
llm_topic_labels[-1] = "Outlier Topic"
topic_model.set_topic_labels(llm_topic_labels)


In [142]:
topic_model.visualize_topics(custom_labels=True)

In [143]:
TOPIC_NUM = 10
topic_model.visualize_barchart(top_n_topics=TOPIC_NUM, height=200, custom_labels=True)


In [144]:
topic_model.visualize_hierarchy(custom_labels=True)

In [145]:
topic_model.visualize_heatmap()

In [146]:
topic_distr, _ = topic_model.approximate_distribution(
    ["Cabin crew service: Felt like a nuisance and was deliberately ignored by male cabin crew."]
)
topic_distr


100%|██████████| 1/1 [00:00<00:00, 151.04it/s]


array([[0.07075881, 0.        , 0.        , 0.13336728, 0.22072053,
        0.        , 0.11106732, 0.        , 0.17218778, 0.        ,
        0.        , 0.04801202, 0.03067747, 0.        , 0.05912506,
        0.        , 0.08070109, 0.07338265, 0.        , 0.        ,
        0.        ]])

In [147]:
topic_model.visualize_distribution(topic_distr[0], custom_labels=True)

## Predict topics for new reviews

In [148]:
topic_distr, topic_token_distr = topic_model.approximate_distribution(all_reviews, calculate_tokens=True)

df = topic_model.visualize_approximate_distribution(all_reviews[1], topic_token_distr[1])
df


100%|██████████| 1/1 [00:00<00:00, 22.66it/s]


,In,flight,smoking,issue,There,was,smoking,on,the,plane
10_amenities_announcement_bottles_spent,0.000,0.000,0.000,0.000,0.102,0.102,0.102,0.102,0.000,0.000
11_catering_trip_complimentary_luggage,0.000,0.109,0.227,0.345,0.345,0.237,0.118,0.000,0.000,0.000


In [149]:
new_document_topic, topic_probabilities = topic_model.transform(
    ["the movie was great but was having some audio glitches along the way which put off the experience"]
)
topic_id = new_document_topic[0]
topic_words = topic_model.get_topic(topic_id)
topic_string = ", ".join([word for word, _ in topic_words])
print(f"The new document is related to Topic {topic_id}: {topic_string}")
print(topic_probabilities)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-26 15:57:38,964 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


The new document is related to Topic 6: flight, the, to, was, cancellation, delayed, delay, before, time, change
[0.55949724]


## Save Model

In [150]:
embedding_model_name = "BAAI/bge-base-en-v1.5"
topic_model.save(
    "my_model_dir",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=embedding_model_name,
)
